# **Building saliency maps: Vanilla gradients and Input x Gradient with Torch**

Welcome to the practical part of the [course](https://open-xai-platform.web.app) devoted to "vanilla gradients" (Vanilla Gradients) and Input x Gradient — the basic methods for explaining deep learning models.

In the [original paper](https://arxiv.org/pdf/1312.6034), the Vanilla Gradients method is proposed for images, so the tutorial focuses exactly on this type of data. By the definition of the method, nothing prevents you from applying it to text, however we have not found practical examples in papers and studies.

Happy coding!

In [ ]:
import torch
import torchvision
from PIL import Image
import numpy as np
import requests
import urllib.request
from io import BytesIO
import matplotlib.pyplot as plt

In [ ]:
# Loading the image
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/pig.png'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes))

image = image.convert("RGB")
plt.imshow(image)
plt.show()

Before working with the image, let us preprocess it (a classical step). Note that we set `mean` and `std` right away, without computing them. For models trained on Imagenet these values are common practice. They were computed from the images of the dataset.

If you want to train from scratch on your own dataset, you can compute a new mean and standard deviation. Otherwise it is recommended to use a pre-validated Imagenet model with its own mean and standard deviation.

In [ ]:
#Means
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Let us preprocess the image
transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),          # Conversion to a tensor
    torchvision.transforms.Normalize(mean, std) # Normalization
])

image = transform(image)

plt.imshow(image.permute((1, 2, 0))); #let us look at what we got after the normalization

After preprocessing the image, in order to visualize readable saliency maps, we would like to get the image back. This is easy to do. Let us recall how normalization works.

Let $X_{1, 2, 3}$ be the image that is to be normalized. Let $M_{1, 2, 3}$, $S_{1, 2, 3}$ be the means and the standard deviations for each channel respectively.

Then $X'_i = \frac{(X_{i}-M_{i})}{S_{i}}$.

Hence $X_i= X'_iS_i+M_i$

Accordingly, for the inverse transformation we need to:
-  multiply by the standard deviation;
- add the means.

In the language of `torch` this is written as follows:

In [ ]:
#A function for performing the inverse transformation
invTrans = torchvision.transforms.Compose([torchvision.transforms.Normalize(mean = [ 0., 0., 0. ],
                                                     std = [ 1/0.229, 1/0.224, 1/0.225 ]),
                                torchvision.transforms.Normalize(mean = [-0.485, -0.456, -0.406 ],
                                                     std = [ 1., 1., 1. ]),
                               ])

inv_original = invTrans(image).squeeze()


plt.imshow(inv_original.permute((1, 2, 0)).detach().numpy()); # let us check that everything works

## **Vanilla backpropagation**

Let us continue working with the transformed image. As a reminder, right now it is stored in the variable `image`.

Suppose we have already solved the task. Let us load a trained model and prepare everything for getting a prediction.

In [ ]:
model = torchvision.models.densenet201(True); #Let us load a trained model
model.eval();

In [ ]:
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

**Quiz:** how many classes is the model able to predict?

In [ ]:
image.unsqueeze_(0);  # let us add the batch dimension

image.requires_grad = True  # we explicitly tell PyTorch to compute and store the gradients for the input image input_img with the .requires_grad flag.

out = model(image)  # we do a forward pass, we get the inference

In [ ]:
# Let us extract the class probabilities from the prediction
probabilities = torch.nn.functional.softmax(out, dim=1)

**Quiz** Among all the classes, extract the class with the highest probability. As the answer, give the predicted probability rounded to two decimal places.

In [ ]:
# Among them let us extract the class with the highest probability

best_id = # Your code here
best_proba = # Your code here

print('Predicted probability:', best_proba)
print('Predicted class:', categories[best_id])

**Quiz** Extract the numbers and the names of the top-3 classes. As the answer, give the name of the class with the lowest probability.

In [ ]:
top_3indicies = # Your code here
top3_names = # Your code here

print(top3_names)

In [ ]:
out[0, best_id].backward() # let us perform a backward pass aka backpropadation
grads = image.grad

## **Building heatmaps**

A heatmap (also known as an activation map) can be visualized in any color scale. In the lesson we discussed how a heatmap is built in the case of a three-channel image. We take the absolute maximum activation over each of the channels and display them on the plane.

In [ ]:
saliency, _ = torch.max(grads.data.abs(),dim=1) # let us build the heatmap

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 12))


ax[0].imshow(saliency[0], cmap=plt.cm.hot)
ax[0].set_title('Saliency Map for class hog')

ax[1].imshow(inv_original.permute((1, 2, 0)).detach().numpy())
ax[1].set_title('Original Image')

for i in range(0, 2):
  ax[i].axis('off')

plt.show()

Also, in some cases it is more convenient to look at the heatmap in grayscale. For this you first need to restore the original colors in the obtained gradients, and then convert them by the [formula](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.convert):

$$I_{gray} = I_r*0.2989 + I_g*0.5780 + I_b*0.1140$$

where $R$, $G$, $B$ are the red, green and blue color channels respectively.

 That is, like this:

In [ ]:
inv_grads = invTrans(grads).squeeze() # we remove the unit dimensions (in our case this is the batch dimension)
inv_grads = inv_grads.permute((1, 2, 0)).detach().numpy()

inv_grads_gray = inv_grads[:,:,0] * 0.2989 + inv_grads[:,:,1] * 0.5780 + inv_grads[:,:,2] * 0.1140 # we sum the values of the color channels with the needed coefficients


#Visualization

plt.imshow(inv_grads_gray, cmap="gray")
plt.title('Grayscale heatmap for class Hog')
plt.axis('off');

Besides that, in some cases it is useful to overlay the map on the image. This can be done with mathematical transformations — a **linear combination** of the source arrays.

**Quiz** Let $X, S$ be the original image and the heatmap respectively, and $\alpha, \beta$ some images. Choose all the options that are linear combinations of $X, S$.

In [ ]:
#The original image in grayscale and the overlaid heatmap

inv_original = inv_original.permute((1, 2, 0)).detach().numpy() # we bring the inversely transformed original into a visualizable form

inv_original_gray = inv_original[:,:,0] * 0.2989 + inv_original[:,:,1] * 0.5780 + inv_original[:,:,2] * 0.1140 #we convert the original to grayscale

plt.imshow(inv_grads_gray+inv_original_gray, cmap='gray')

In [ ]:
# Feel free to play with the coefficient!
for i in range(1, 10, 2):
  plt.imshow(inv_grads_gray*i+inv_original_gray, cmap='gray')
  plt.title(f'Overlay with coefficient {i}')
  plt.show()

In [ ]:
#  The same can be done with the image in color

plt.imshow(inv_original/2 + grads.data.squeeze().abs().permute((1, 2, 0)).detach().numpy()*5);
plt.title('Overlay of the gradients on the original image in color')
plt.show()

## Input x Gradient
Let us move on to the next method we have studied. By the definition from the lesson, the Input x Gradient heatmap is: $$GradientXInput = x\odot \nabla{F_c(x)}$$

So, to get such a map we can go two ways:
1.  use a ready-made method from a library
2.  build it on the basis of Vanilla Gradient

Let us look at both methods one by one.

## Captum

Captum is an explainable AI library for models created with PyTorch. It contains the whole main set of existing and used interpretation methods, which is why throughout the course we will often use exactly this library.


> "Captum" in Latin means "comprehension".

In [ ]:
!pip install captum -q

In [ ]:
from captum.attr import InputXGradient

input_x_gradient_captum = InputXGradient(model)

attribution = input_x_gradient_captum.attribute(image, target=best_id) # Building the map

Let us visualize the inputXgradient map for the class with the best id. As a reminder, for us this is id 341.

In [ ]:
# Visualization of the class obtained with InputXGradient from the library

plt.imshow(attribution.squeeze().permute((1, 2, 0)).detach().numpy())
plt.axis('off')

plt.title('Input x Gradient saliency map')
plt.show();

Now let us turn to the definition and compute the map by hand. The values stored as grads reflect the matrix $\nabla{F_c(x)}$.

**Quiz** complete the code to build the map by the definition. As the answer, write down the value stored at the coordinate input_x_grad_hand_one[1, 7, 7] (the first channel, the seventh row, the seventh column).

In [ ]:
input_x_grad_hand_one = # Your code here

plt.imshow(input_x_grad_hand_one.permute(1, 2, 0).detach().numpy())

In [ ]:
input_x_grad_hand_one[1, 7, 7]

**Quiz** Compare coordinate by coordinate the values at the coordinates of `attribution` and `input_x_grad_hand_one`. Are they equal?

In [ ]:
# Your code here

# **Additional materials**:
- [A tutorial](https://www.coderskitchen.com/explainable-ai-how-to-implement-saliency-maps/) on creating saliency maps, if you work with tensorflow